# **Build a Dashboard Application with Plotly Dash**

In this lab, you will be building a Plotly Dash application for users to perform interactive visual analytics on SpaceX launch data in real-time.

This dashboard application contains input components such as a dropdown list and a range slider to interact with a pie chart and a scatter point chart. You will be guided to build this dashboard application via the following tasks:
- **TASK 1:** Add a Launch Site Drop-down Input Component
- **TASK 2:** Add a callback function to render `success-pie-chart` based on selected site dropdown
- **TASK 3:** Add a Range Slider to Select Payload
- **TASK 4:** Add a callback function to render the `success-payload-scatter-chart` scatter plot

*Note: this lab was originally provided as a standalone Python script (`spacex_dash_app.py`) run from a terminal in IBM's Cloud IDE, rather than a Jupyter notebook. It has been rebuilt here as a notebook for consistency with the rest of this project's structure on GitHub. To actually run the dashboard, either export the final cells below to a `.py` file and run `python spacex_dash_app.py` from a terminal, or run this notebook using `jupyter-dash`/`app.run(mode='inline')` if working in an environment that supports it.*

## Setup: import required libraries and load the dataset

In [1]:
# Import required libraries
import pandas as pd
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output
import plotly.express as px

Download the SpaceX launch dataset used for this dashboard (`spacex_launch_dash.csv`), then load it into a dataframe.

In [2]:
# Read the SpaceX launch data into a pandas dataframe
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()
spacex_df.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [3]:
# Create a dash application
app = dash.Dash(__name__)

## TASK 1: Add a Launch Site Drop-down Input Component

We have four different launch sites and we would like to first see which one has the largest success count. Then, we would like to select one specific site and check its detailed success rate (`class=0` vs. `class=1`).

As such, we will need a dropdown menu to let us select different launch sites.

Find and complete a commented `dcc.Dropdown(id='site-dropdown',...)` input with the following attributes:
- `id` attribute with value `site-dropdown`
- `options` attribute is a list of dict-like option objects (with `label` and `value` attributes). You can set the `label` and `value` all to be the launch site names in `spacex_df`, and you need to include the default `All` option, e.g.:
```python
options=[{'label': 'All Sites', 'value': 'ALL'}, {'label': 'site1', 'value': 'site1'}, ...]
```
- `value` attribute with default dropdown value to be `ALL`, meaning all sites are selected
- `placeholder` attribute to show a text description about this input area, such as `Select a Launch Site here`
- `searchable` attribute to be `True` so we can enter keywords to search launch sites

Build the app layout, including the Task 1 dropdown component:

In [10]:
# Create an app layout
app.layout = html.Div(children=[html.H1('SpaceX Launch Records Dashboard',
                                        style={'textAlign': 'center', 'color': '#503D36',
                                               'font-size': 40}),

                                # TASK 1: Add a dropdown list to enable Launch Site selection
                                # The default select value is for ALL sites
                                dcc.Dropdown(id='site-dropdown',
                                             options=[
                                             {'label': 'All Sites', 'value': 'ALL'},
                                             {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
                                             {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},
                                             {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
                                             {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}
                                             ],
                                             value='ALL',
                                             placeholder="Select a Launch Site here",
                                             searchable=True
                                             ),

                                html.Br(),

                                # TASK 2: Add a pie chart to show the total successful launches count for all sites
                                # If a specific launch site was selected, show the Success vs. Failed counts for the site
                                html.Div(dcc.Graph(id='success-pie-chart')),
                                html.Br(),

                                html.P("Payload range (Kg):"),
                                # TASK 3: Add a slider to select payload range
                                dcc.RangeSlider(id='payload-slider', 
                                               min=0, max=10000, step=1000,
                                               marks={0: '0', 2500: '2500', 5000: '5000', 7500: '7500', 10000: '10000'},
                                               value=[min_payload, max_payload]),

                                # TASK 4: Add a scatter chart to show the correlation between payload and launch success
                                html.Div(dcc.Graph(id='success-payload-scatter-chart')),
                                ])

## TASK 2: Add a callback function to render `success-pie-chart` based on selected site dropdown

The general idea of this callback function is to get the selected launch site from `site-dropdown` and render a pie chart visualizing launch success counts.

Dash callback functions are automatically called by Dash whenever the input component updates, such as a click or dropdown selecting event.

Let's add a callback function including the following application logic:
- Input is set to be the `site-dropdown` dropdown, i.e. `Input(component_id='site-dropdown', component_property='value')`
- Output to be the graph with id `success-pie-chart`, i.e. `Output(component_id='success-pie-chart', component_property='figure')`
- An if-else statement to check if `ALL` sites were selected or just a specific launch site was selected
    - If `ALL` sites are selected, use all rows in `spacex_df` to render a pie chart showing the total success launches (i.e. the total count of the `class` column)
    - If a specific launch site is selected, filter `spacex_df` first to include only that site's data, then render a pie chart showing the success (`class=1`) and failed (`class=0`) counts for that site

In [11]:
# Function decorator to specify function input and output
@app.callback(Output(component_id='success-pie-chart', component_property='figure'),
              Input(component_id='site-dropdown', component_property='value'))
def get_pie_chart(entered_site):
    filtered_df = spacex_df
    if entered_site == 'ALL':
        fig = px.pie(filtered_df, values='class', 
                     names='Launch Site', 
                     title='Total Success Launches by Site')
        return fig
    else:
        site_df = spacex_df[spacex_df['Launch Site'] == entered_site]
        fig = px.pie(site_df, names='class', 
                     title=f'Total Success vs. Failure Launches for {entered_site}')
        return fig

## TASK 3: Add a Range Slider to Select Payload

Next, we want to find if the variable payload is correlated to mission outcome. From a dashboard point of view, we want to be able to easily select different payload ranges and see if we can identify some visual patterns.

Find and complete a commented `dcc.RangeSlider(id='payload-slider',...)` input with the following attributes:
- `id` to be `payload-slider`
- `min` indicating the slider starting point, set to `0` (Kg)
- `max` indicating the slider ending point, set to `10000` (Kg)
- `step` indicating the slider interval, set to `1000` (Kg)
- `value` indicating the current selected range, set to `[min_payload, max_payload]`

*Note: this component has been completed in Task 1 above in the # Task 3 section

## TASK 4: Add a callback function to render the `success-payload-scatter-chart` scatter plot

Next, we want to plot a scatter plot with the x axis as the payload and the y axis as the launch outcome (i.e. the `class` column), so we can visually observe how payload may be correlated with mission outcomes for the selected site(s).

We also want to colour-label the Booster version on each scatter point, so we can observe mission outcomes across different boosters.

Let's add a callback function including the following application logic:
- Input to be `[Input(component_id='site-dropdown', component_property='value'), Input(component_id="payload-slider", component_property="value")]` — two input components, one for the selected launch site and one for the selected payload range
- Output to be `Output(component_id='success-payload-scatter-chart', component_property='figure')`
- An if-else statement to check if `ALL` sites were selected or just a specific launch site was selected
    - If `ALL` sites are selected, render a scatter plot for all values of `Payload Mass (kg)` vs. `class`, colour-labelled by `Booster Version Category`
    - If a specific launch site is selected, filter `spacex_df` first, then render the scatter chart for that site's `Payload Mass (kg)` vs. `class`, colour-labelled by `Booster Version Category`

In [12]:
# TASK 4: Add a callback function for `site-dropdown` and `payload-slider` as inputs, `success-payload-scatter-chart` as output
@app.callback(Output(component_id='success-payload-scatter-chart', component_property='figure'),
              [Input(component_id='site-dropdown', component_property='value'),
               Input(component_id="payload-slider", component_property="value")])
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    mask = (spacex_df['Payload Mass (kg)'] > low) & (spacex_df['Payload Mass (kg)'] < high)
    filtered_df = spacex_df[mask]
    if entered_site == 'ALL':
        fig = px.scatter(filtered_df, x='Payload Mass (kg)', y='class',
                          color='Booster Version Category',
                          title='Correlation between Payload and Success for all Sites')
        return fig
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        fig = px.scatter(site_df, x='Payload Mass (kg)', y='class',
                          color='Booster Version Category',
                          title=f'Correlation between Payload and Success for site {entered_site}')
        return fig

## Finding Insights Visually

Now with the dashboard completed, you should be able to use it to analyze SpaceX launch data, and answer the following questions:
1. Which site has the largest successful launches?
KSC LC-39A (10 launches)
2. Which site has the highest launch success rate?
CCFAS SLC-40 (42.9%)
3. Which payload range(s) has the highest launch success rate?
The 2-4k payload range has the highest success rate (60%)
4. Which payload range(s) has the lowest launch success rate?
The 6-8k payload range has the lowest success rate (0%)
5. Which F9 Booster version (`v1.0`, `v1.1`, `FT`, `B4`, `B5`, etc.) has the highest launch success rate?
B5 has the highest success rate as it only has one flight which was successful, therefore, the success rate is 100%/

*Note: Please take screenshots of the completed dashboard and save them — the GitHub URL and screenshots are required later for the presentation slides.*

## Run the app

In [14]:
# Run the app
# If running as a standalone script (python spacex_dash_app.py), this starts a local server, typically at http://127.0.0.1:8050/
if __name__ == '__main__':
    app.run()

## Authors

Yan Luo

### Other Contributors

Joseph Santarcangelo